# <u>Group S</u>: Anne Faury, Héloise Lordez, Cyprien Charlaté, Lucas Gorrec, William Roose


# Week 2 — Baby Names: Initial Implementation

This notebook explore three interactive visualisations made with **Altair 6** exploring the French baby-names dataset (department agregation, year-by-year, 1900-2020). We implement here our three favorite designs chosen during the first week. The given notebook from the 'Names hints' folder and previous class labs on Altair give us some inspiration and tools to build the current notebook.

To get a further understanding and explanations about our results, check our post on the forum on Moodle.

### Reminder:
<u>Visualization 1</u>: 
- How do baby names evolve over time?
- Are there names that have consistently remained popular or unpopular?
- Are there some that have were suddenly or briefly popular or unpopular?
- Are there trends in time?

<u>Visualization 2</u>: 
- Is there a regional effect in the data? 
- Are some names more popular in some regions? 
- Are popular names generally popular across the whole country?

<u>Visualization 3</u>: 
- Are there gender effects in the data? 
- Does popularity of names given to both sexes evolve consistently? (Note: this data set treats sex as binary; this is a simplification that carries into this assignment but does not generally hold.)


> **Data notes**  
> - Rows with `preusuel == "_PRENOMS_RARES"` (rare names) are dropped.  
> - Rows with `dpt == "XX"` (unknown department) and `annais == "XXXX"` (unknown year) are dropped.  
> - `sexe`: 1 = male, 2 = female.

In [ ]:
import altair as alt
import pandas as pd
import geopandas as gpd

# html renderer outputs interactive Vega-Embed HTML (text/html) that VS Code renders natively.
# disable_max_rows inlines all data so no external file references are needed.
alt.data_transformers.disable_max_rows()
alt.renderers.enable('html')

DATA  = "../Names hints/dpt2020.csv"
GEO_M = "../Names hints/departements-avec-outre-mer.geojson"

# ── Load & clean ──────────────────────────────────────────────────────────────
df = pd.read_csv(DATA, sep=";", dtype={"dpt": str, "annais": str})
df = df[
    (df.preusuel != "_PRENOMS_RARES") &
    (df.dpt      != "XX")             &
    (df.annais   != "XXXX")
]
df["annais"] = df["annais"].astype(int)
df["nombre"] = df["nombre"].astype(int)
df["sexe"]   = df["sexe"].astype(int)

print(f"Rows: {len(df):,}  |  Unique names: {df.preusuel.nunique():,}  "
      f"|  Years: {df.annais.min()}–{df.annais.max()}  "
      f"|  Departments: {df.dpt.nunique()}")
df.head(3)

## Visualization 1 — How do baby names evolve over time?

**Design:** A multi-line time-series chart of the **20 most popular names** at the national level (sum over all departments and both sexes). Clicking a name **in the legend** highlights its line and dims all others, making it easy to compare trajectories or spot brief fashions vs. steady favourites.

**Why this works:**
- The full 1900–2020 range immediately reveals the effects of wars, film stars, and fashion cycles.
- The dimming interaction cuts through the clutter of 20 simultaneous lines.
- Hovering shows exact year + count.

**Strengths:** Shows temporal context for many names simultaneously; click-to-highlight is intuitive.  
**Weaknesses:** Limited to the pre-selected top 20; names outside that set are invisible; 20 colours are hard to distinguish at a glance.

In [ ]:
# ── Data prep ─────────────────────────────────────────────────────────────────
national   = df.groupby(["preusuel", "annais"], as_index=False)["nombre"].sum()
top20_names = national.groupby("preusuel")["nombre"].sum().nlargest(20).index.tolist()
data_v1    = national[national.preusuel.isin(top20_names)].copy()

# ── Chart ─────────────────────────────────────────────────────────────────────
sel = alt.selection_point(fields=["preusuel"], bind="legend")

v1 = (
    alt.Chart(data_v1)
    .mark_line()
    .encode(
        x=alt.X("annais:Q", title="Year",
                axis=alt.Axis(format="d", tickCount=13)),
        y=alt.Y("nombre:Q", title="Number of births (national)"),
        color=alt.Color("preusuel:N",
                        legend=alt.Legend(title="Click a name to highlight",
                                          symbolStrokeWidth=3)),
        opacity=alt.condition(sel, alt.value(1.0), alt.value(0.07)),
        strokeWidth=alt.condition(sel, alt.value(3.5), alt.value(1.0)),
        tooltip=[
            alt.Tooltip("preusuel:N", title="Name"),
            alt.Tooltip("annais:Q",   title="Year",   format="d"),
            alt.Tooltip("nombre:Q",   title="Births", format=","),
        ],
    )
    .add_params(sel)
    .properties(
        title="National births per year (1900-2020) - Top-20 French baby names",
        width=800,
        height=430,
    )
)

v1

## Visualization 2 — Is there a regional effect?

**Design:** A **choropleth map** of metropolitan France showing, for a name chosen from a dropdown, the proportion of births with that name per 1 000 total births in each department.

Normalising by total departmental births removes the population-size bias: a peak in a small rural department is just as visible as one in Île-de-France.

**Implementation trick:** The top-30 names are pivoted into a **wide table** (one column per name, one row per department) before merging with the GeoJSON. A `transform_fold` + `transform_filter` in Altair then selects the relevant column in the browser — avoiding the 30× geometry duplication that a long table would cause.

**Strengths:** Spatially intuitive; proportion encoding is fair regardless of department size.  
**Weaknesses:** Dropdown limited to top-30 names; overseas territories not shown.

In [ ]:
import warnings
import math

# ── 1. National top-3 per year ───────────────────────────────────────────────
nat_v2 = df.groupby(["preusuel", "annais"], as_index=False)["nombre"].sum()
nat_v2["rank_id"] = (
    nat_v2.groupby("annais")["nombre"]
    .rank(method="first", ascending=False)
    .astype(int)
)
top3_yr       = nat_v2[nat_v2["rank_id"] <= 3][["preusuel", "annais", "rank_id"]].copy()
top3_names_v2 = top3_yr["preusuel"].unique().tolist()

# ── 2. Dept proportions (per 1 000 births) ───────────────────────────────────
by_dpt = (
    df[df["preusuel"].isin(top3_names_v2)]
    .groupby(["preusuel", "dpt", "annais"], as_index=False)["nombre"].sum()
)
tot_dpt = (
    df.groupby(["dpt", "annais"], as_index=False)["nombre"]
    .sum().rename(columns={"nombre": "total"})
)
by_dpt = by_dpt.merge(tot_dpt, on=["dpt", "annais"])
by_dpt["prop"] = (by_dpt["nombre"] / by_dpt["total"] * 1_000).round(1)
data_long_v2 = (
    top3_yr.merge(by_dpt, on=["preusuel", "annais"], how="left")
    .fillna({"prop": 0.0})
)

# ── 3. Wide pivot: 1 row per (dept × rank_id), columns p_YYYY ────────────────
def rank_wide(rid):
    sub = data_long_v2[data_long_v2["rank_id"] == rid]
    w = (
        sub.pivot_table(index="dpt", columns="annais", values="prop", aggfunc="first")
        .fillna(0)
    )
    w.columns = [f"p_{int(y)}" for y in w.columns]
    w = w.reset_index()
    w["rank_id"] = rid
    return w

wide_all_v2 = pd.concat([rank_wide(1), rank_wide(2), rank_wide(3)], ignore_index=True)

# INSEE CSV codes Corsica as "20"; GeoJSON uses "2A"/"2B" → duplicate data rows
_corse = wide_all_v2[wide_all_v2["dpt"] == "20"].copy()
if len(_corse) > 0:
    _c2a = _corse.copy(); _c2a["dpt"] = "2A"
    _c2b = _corse.copy(); _c2b["dpt"] = "2B"
    wide_all_v2 = pd.concat(
        [wide_all_v2[wide_all_v2["dpt"] != "20"], _c2a, _c2b],
        ignore_index=True,
    )

# ── 4. Merge geometry — DOM-TOM repositioned + latitude correction ────────────
from shapely.affinity import translate as _sha_translate, scale as _sha_scale

depts_geo_v2 = gpd.read_file(GEO_M)

# DOM-TOM placed just below France's southern coast (y ≈ 40.4-40.6°)
_inset_cfg = {
    # code : (target_center_x, target_center_y, target_size_degrees)
    "971": (-4.0, 40.6, 0.8),   # Guadeloupe
    "972": (-1.5, 40.6, 0.6),   # Martinique
    "973": ( 1.5, 40.4, 0.8),   # Guyane
    "974": ( 6.0, 40.6, 0.8),   # La Réunion
    "976": ( 8.8, 40.6, 0.5),   # Mayotte (no data in CSV → won't appear in geo_v2)
}

def _reposition(gdf, cfg):
    rows = []
    for _, row in gdf.iterrows():
        if row["code"] not in cfg:
            rows.append(row)
            continue
        b = row.geometry.bounds
        orig_size = max(b[2] - b[0], b[3] - b[1])
        tx, ty, tsize = cfg[row["code"]]
        factor = tsize / orig_size
        g = _sha_scale(row.geometry, xfact=factor, yfact=factor, origin="centroid")
        b2 = g.bounds
        g = _sha_translate(g, tx - (b2[0]+b2[2])/2, ty - (b2[1]+b2[3])/2)
        row = row.copy()
        row["geometry"] = g
        rows.append(row)
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=gdf.crs)

depts_geo_v2 = _reposition(depts_geo_v2, _inset_cfg)

# Latitude correction: stretch y by 1/cos(46°) ≈ 1.441 so France renders in
# correct proportions with identity projection (1° lon ≠ 1° lat in km at 46°N).
_LAT_CORR = 1.0 / math.cos(math.radians(46.0))
depts_geo_v2["geometry"] = depts_geo_v2.geometry.apply(
    lambda g: _sha_scale(g, xfact=1.0, yfact=_LAT_CORR, origin=(0, 0))
)
depts_geo_v2 = depts_geo_v2.set_geometry("geometry")

geo_v2 = depts_geo_v2.merge(wide_all_v2, how="inner", left_on="code", right_on="dpt")
geo_v2 = geo_v2.drop(columns=["dpt"])

global_max_v2 = float(data_long_v2["prop"].max())
name_src_v2   = top3_yr[["preusuel", "annais", "rank_id"]].copy()
print(f"geo_v2: {len(geo_v2)} rows | colour domain [0, {global_max_v2:.1f}]")

# ── 5. Params ─────────────────────────────────────────────────────────────────
year_p_v2 = alt.param(
    name="v2_year",
    bind=alt.binding_range(min=1900, max=2020, step=1, name="Year : "),
    value=2000,
)
click_sel_v2 = alt.selection_point(
    fields=["rank_id"],
    name="v2_click",
    empty=False,
    value=[{"rank_id": 1}],
)

# ── 6. Layout constants ───────────────────────────────────────────────────────
# After lat-correction: bbox ≈ 15° wide × 16° tall → LH = LW × 16/15 ≈ 565
SW, SH = 205, 220   # small map
LW, LH = 530, 565   # large map

# No fixed domain → Vega-Lite auto-scales on the visible (year-filtered) data,
# giving maximum geographic contrast for the selected year.
COLOR_SMALL = alt.Color(
    "prop:Q",
    scale=alt.Scale(scheme="blues"),
    legend=None,
)
COLOR_LARGE = alt.Color(
    "prop:Q",
    scale=alt.Scale(scheme="blues"),
    legend=alt.Legend(title="For 1 000 births"),
)

# ── 7. Name labels — medal + name, blue when selected ────────────────────────
_MEDALS = {1: "🥇", 2: "🥈", 3: "🥉"}

def name_label_v2(rid, w):
    selected = f"v2_click.rank_id == {rid}"
    medal = _MEDALS[rid]
    return (
        alt.Chart(name_src_v2)
        .mark_text(align="center", baseline="middle", fontSize=13, fontWeight="bold")
        .encode(
            text="label:N",
            x=alt.value(w / 2),
            y=alt.value(12),
            color=alt.condition(selected, alt.value("#1a6bb5"), alt.value("#aaaaaa")),
        )
        .transform_filter(f"datum.annais == v2_year && datum.rank_id == {rid}")
        .transform_calculate("label", f"'{medal} ' + datum.preusuel")
        .properties(width=w, height=22)
    )

# ── 8. Small maps — full opacity always, click to select rank ─────────────────
def small_map_v2(rid):
    selected = f"v2_click.rank_id == {rid}"
    return (
        alt.Chart(geo_v2)
        .mark_geoshape(stroke="white", strokeWidth=0.4, cursor="pointer")
        .encode(
            color=COLOR_SMALL,
            tooltip=[
                alt.Tooltip("nom:N",  title="Department"),
                alt.Tooltip("prop:Q", title="Per 1 000", format=".1f"),
            ],
        )
        .transform_calculate("prop", "datum['p_' + v2_year]")
        .transform_filter(f"datum.rank_id == {rid}")
        .add_params(click_sel_v2)
        .project(type="identity", reflectY=True)
        .properties(width=SW, height=SH)
    )

# ── 9. Large map — legend on the right ───────────────────────────────────────
large_label_v2 = (
    alt.Chart(name_src_v2)
    .mark_text(align="center", baseline="middle", fontSize=15, fontWeight="bold")
    .encode(text="preusuel:N", x=alt.value(LW / 2), y=alt.value(12))
    .transform_filter("datum.annais == v2_year && datum.rank_id == v2_click.rank_id")
    .properties(width=LW, height=22)
)

large_map_v2 = (
    alt.Chart(geo_v2)
    .mark_geoshape(stroke="white", strokeWidth=0.5)
    .encode(
        color=COLOR_LARGE,
        tooltip=[
            alt.Tooltip("nom:N",  title="Department"),
            alt.Tooltip("prop:Q", title="Per 1 000", format=".1f"),
        ],
    )
    .transform_calculate("prop", "datum['p_' + v2_year]")
    .transform_filter("datum.rank_id == v2_click.rank_id")
    .project(type="identity", reflectY=True)
    .properties(
        width=LW, height=LH,
        title=alt.TitleParams(
            "Detailed view: ",
            subtitle="Click on a mini map above to view in it full size",
            subtitleFontSize=11,
            subtitleColor="gray",
        ),
    )
)

# ── 10. Assemble ──────────────────────────────────────────────────────────────
labels_row_v2 = alt.hconcat(
    name_label_v2(1, SW), name_label_v2(2, SW), name_label_v2(3, SW), spacing=2
)
small_row_v2 = alt.hconcat(
    small_map_v2(1), small_map_v2(2), small_map_v2(3), spacing=10
).resolve_scale(color="independent")

# Thin horizontal rule between the small maps and the detailed view
separator_v2 = (
    alt.Chart(pd.DataFrame({"y": [0]}))
    .mark_rule(color="#222222", strokeWidth=1.5)
    .encode(y=alt.value(1))
    .properties(width=SW * 3 + 15, height=4)
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    v2_new = (
        alt.vconcat(
            labels_row_v2,
            small_row_v2,
            separator_v2,
            large_label_v2,
            large_map_v2,
            spacing=10,
        )
        .resolve_scale(color="independent")
        .add_params(year_p_v2)
        .configure_view(strokeWidth=0)
    )

v2_new


## Visualization 3 — Is there a gender effect on unisex names?

**Design:** A **butterfly chart** (diverging horizontal bar chart) for a unisex name selected from a dropdown. Each year is a row on the vertical axis (ascending). Boys' bars extend **left**, girls' bars extend **right**, with a vertical centre line at zero.

Three controls let the user explore the data:
- **First name** dropdown (top-50 unisex names by total popularity)
- **From / To year** sliders to zoom into a period of interest
- **Step (years)** slider to thin out the bars (e.g. one bar every 5 years)

**Why this works:**
- The mirror layout immediately reveals whether the name is predominantly male, female, or truly balanced.
- Because boys = left and girls = right, the *total* number of babies named X in a given year is simply the sum of both bar lengths — readable at a glance.
- Changing the time range or step lets the user trace how the gender balance shifts decade by decade.

**Strengths:** Directly answers the gender-effect question; raw counts allow easy total estimation; controls make any time window explorable.  
**Weaknesses:** Only names used for both sexes are shown; very lopsided names may have one near-invisible bar.

In [ ]:
# ── Data prep ─────────────────────────────────────────────────────────────────
nat_sex_v3 = df.groupby(["preusuel", "annais", "sexe"], as_index=False)["nombre"].sum()

# Unisex names: given to both sexes with ≥ 200 total births per sex
sex_tot_v3 = nat_sex_v3.groupby(["preusuel", "sexe"])["nombre"].sum().reset_index()
sex_piv_v3 = sex_tot_v3.pivot(index="preusuel", columns="sexe", values="nombre").fillna(0)
dual_v3    = sex_piv_v3[(sex_piv_v3[1] >= 200) & (sex_piv_v3[2] >= 200)].index.tolist()
dual_sorted_v3 = (
    df[df.preusuel.isin(dual_v3)]
    .groupby("preusuel")["nombre"].sum()
    .sort_values(ascending=False)
    .head(50).index.tolist()
)
_pref_v3   = ["DOMINIQUE", "CAMILLE", "CLAUDE"]
default_v3 = next((n for n in _pref_v3 if n in dual_sorted_v3), dual_sorted_v3[0])
data_v3    = nat_sex_v3[nat_sex_v3.preusuel.isin(dual_sorted_v3)].copy()
name_df_v3 = pd.DataFrame({"preusuel": dual_sorted_v3})
print(f"Unisex names in dropdown: {len(dual_sorted_v3)}.  Default: {default_v3}")

# ── Params ────────────────────────────────────────────────────────────────────
name_p3 = alt.param(
    name="v3_name",
    bind=alt.binding_select(options=dual_sorted_v3, name="First name: "),
    value=default_v3,
)
ymin_p3 = alt.param(
    name="v3_ymin",
    bind=alt.binding_range(min=1900, max=2019, step=1, name="From year: "),
    value=1970,
)
ymax_p3 = alt.param(
    name="v3_ymax",
    bind=alt.binding_range(min=1901, max=2020, step=1, name="To year: "),
    value=2020,
)
step_p3 = alt.param(
    name="v3_step",
    bind=alt.binding_range(min=1, max=20, step=1, name="Step (years): "),
    value=5,
)

# ── Chart ─────────────────────────────────────────────────────────────────────
# Dynamic name header
name_title_v3 = (
    alt.Chart(name_df_v3)
    .mark_text(align="center", baseline="middle", fontSize=15, fontWeight="bold")
    .encode(text="preusuel:N", x=alt.value(300), y=alt.value(12))
    .transform_filter("datum.preusuel === v3_name")
    .properties(width=600, height=22)
)

# Horizontal butterfly bars
# Boys  → val = -nombre  (extends left)
# Girls → val = +nombre  (extends right)
bars_v3 = (
    alt.Chart(data_v3)
    .mark_bar()
    .encode(
        x=alt.X(
            "val:Q",
            title="← Boys | Girls →",
            axis=alt.Axis(
                labelExpr="format(abs(datum.value), ',.0f')",
                labelAngle=0,
            ),
        ),
        y=alt.Y(
            "annais:O",
            title="Year",
            sort="ascending",
            axis=alt.Axis(labelAngle=0),
        ),
        color=alt.Color(
            "gender:N",
            scale=alt.Scale(domain=["Boys", "Girls"], range=["#4e78a8", "#e7729a"]),
            legend=alt.Legend(title="Sex"),
        ),
        tooltip=[
            alt.Tooltip("annais:O",  title="Year"),
            alt.Tooltip("gender:N",  title="Sex"),
            alt.Tooltip("nombre:Q",  title="Births", format=","),
        ],
    )
    .transform_filter("datum.preusuel === v3_name")
    .transform_filter("datum.annais >= v3_ymin && datum.annais <= v3_ymax")
    .transform_filter("(datum.annais - v3_ymin) % v3_step === 0")
    .transform_calculate("val",    "datum.sexe === 1 ? -datum.nombre : datum.nombre")
    .transform_calculate("gender", "datum.sexe === 1 ? 'Boys' : 'Girls'")
    .properties(width=600, height=400)
)

# Vertical centre line at x = 0
centre_v3 = (
    alt.Chart(pd.DataFrame({"x": [0]}))
    .mark_rule(color="#333333", strokeWidth=1.5)
    .encode(x=alt.X("x:Q"))
)

v3 = (
    alt.vconcat(
        name_title_v3,
        alt.layer(bars_v3, centre_v3),
        spacing=6,
    )
    .add_params(name_p3, ymin_p3, ymax_p3, step_p3)
    .properties(
        title=alt.TitleParams(
            "Gender butterfly chart - Boys vs Girls",
            subtitle="Raw birth counts. Work on top-50 unisex names by total popularity (1900-2020)",
            subtitleFontSize=11,
            subtitleColor="gray",
        )
    )
    .configure_view(strokeWidth=0)
)

v3
